In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer
import config

SQUAD_TRAIN_NUM  = 40
SQUAD_VAL_NUM    = 40
SQUAD_TEST_NUM   = 40

SQUAD_METRICS = [
    {"name": "exact_match",         "weight": 0.25, "stage": 1},  # строгое точное совпадение
    {"name": "token_f1",            "weight": 0.25, "stage": 1},  # лексическое перекрытие токенов
    {"name": "no_answer_accuracy",  "weight": 0.20, "stage": 1},  # корректность на unanswerable
    {"name": "semantic_similarity",  "weight": 0.10, "stage": 3},  # семантическая близость
    {"name": "faithfulness",         "weight": 0.10, "stage": 3},  # верность контексту
    {"name": "robustness",           "weight": 0.10, "stage": 3},  # устойчивость
]
config.METRICS_CONFIG = SQUAD_METRICS
config.METRIC_WEIGHTS = {m["name"]: m["weight"] for m in SQUAD_METRICS}

# ─── ЛОКАЛЬНАЯ ОПТИМИЗАЦИЯ ─────────────────────────────────────
config.LOCAL_ITERATIONS_PER_GENERATION = 3
config.LOCAL_CANDIDATES_PER_ITERATION  = 4
config.LOCAL_BATCH_SIZE                = 15
config.LOCAL_PARENTS_PER_ITERATION     = 2
config.MAX_GRADIENT_PAIRS              = 3
config.MINI_BATCH_RATIO                = 0.5
config.PRE_SCREEN_TOP_K                = 4
config.TRAIN_FAILURE_SAMPLE_SIZE       = 40

# ─── ГЛОБАЛЬНАЯ ОПТИМИЗАЦИЯ ────────────────────────────────────
config.GLOBAL_CANDIDATES              = 6
config.GLOBAL_TRIGGER_INTERVAL        = 2
config.GLOBAL_QUALITY_GATE_TOLERANCE  = 0.9
config.GLOBAL_PRESCREEN_GATE          = 0.85
config.GLOBAL_OPTIMIZER_TEMPERATURE   = 0.8
config.CROSSOVER_CANDIDATES           = 3
config.EXEMPLAR_COUNT                 = 7
config.FEW_SHOT_COUNT                 = 5
config.HISTORY_SCORE_THRESHOLD        = 0.3
config.GLOBAL_HISTORY_WINDOW          = 30

# ─── ПОПУЛЯЦИЯ И РАННЯЯ ОСТАНОВКА ──────────────────────────────
config.MAX_GENERATIONS                = 5
config.POPULATION_SIZE                = 3
config.PATIENCE                       = 2
config.FORCE_GLOBAL_AFTER_STAGNATION  = 1
config.MIN_IMPROVEMENT                = 0.005
config.SIMILARITY_THRESHOLD           = 0.85
config.DIVERSITY_WEIGHT               = 0.10

# ─── LLM ───────────────────────────────────────────────────────
config.TEMPERATURE                    = 0.2
config.MAX_TOKENS                     = 3000

# ─── ОЦЕНКА ────────────────────────────────────────────────────
config.MAX_EXAMPLES_PER_NODE          = 100
config.BATCH_EVAL_SIZE                = 50
config.CORRECTNESS_TOKEN_F1_THRESHOLD = 0.4
config.USE_CONTAINMENT_CHECK          = True
config.USE_LLM_CORRECTNESS_CHECK      = False

# ─── СТАБИЛЬНОСТЬ ──────────────────────────────────────────────
config.STABILITY_LAMBDA               = 0.05
config.BOOTSTRAP_RUNS                 = 3

print("SQuAD v2 config applied")
print(f"   Metrics: {[m['name'] for m in SQUAD_METRICS]}")
print(f"   Weights: {config.METRIC_WEIGHTS}")
print(f"   Data splits: train={SQUAD_TRAIN_NUM}, val={SQUAD_VAL_NUM}, test={SQUAD_TEST_NUM}")
print(f"   Generations: {config.MAX_GENERATIONS}, Population: {config.POPULATION_SIZE}")
print(f"   Global candidates: {config.GLOBAL_CANDIDATES}, Gate: {config.GLOBAL_QUALITY_GATE_TOLERANCE}")
print(f"   Temperature: {config.TEMPERATURE}")

/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

SQuAD v2 config applied
   Metrics: ['exact_match', 'token_f1', 'no_answer_accuracy', 'semantic_similarity', 'faithfulness', 'robustness']
   Weights: {'exact_match': 0.25, 'token_f1': 0.25, 'no_answer_accuracy': 0.2, 'semantic_similarity': 0.1, 'faithfulness': 0.1, 'robustness': 0.1}
   Data splits: train=40, val=40, test=40
   Generations: 5, Population: 3
   Global candidates: 6, Gate: 0.9
   Temperature: 0.2


In [2]:
def squad_v2_to_examples(data):
    examples = []
    for item in data:
        input_text = (f'# Question: \n {item["question"]} \n'
                      f'# Context: \n {item["context"]} \n')
        expected_output = item['answers']['text'][0] if item['answers']['text'] else 'No answer'
        examples.append(Example(input_text=input_text, expected_output=expected_output))
    return examples

def get_squad_v2_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('rajpurkar/squad_v2', split='train')
    ds_test = load_dataset('rajpurkar/squad_v2', split='validation')

    split = ds_train.train_test_split(test_size=0.1, seed=42)
    ds_train = split['train']
    ds_val = split['test']

    # ds_train = ds_train.shuffle()
    # ds_val = ds_val.shuffle()
    # ds_test = ds_test.shuffle()

    train_split = ds_train.select(range(train_num))
    val_split = ds_val.select(range(val_num))
    test_split = ds_test.select(range(test_num))

    train_examples = squad_v2_to_examples(train_split)
    validation_examples = squad_v2_to_examples(val_split)
    test_examples = squad_v2_to_examples(test_split)

    return train_examples, validation_examples, test_examples

def data_fabric(dataset: str = 'squad_v2', train_num: int = SQUAD_TRAIN_NUM, val_num: int = SQUAD_VAL_NUM, test_num: int = SQUAD_TEST_NUM):
    squad_v2_initial_prompt = """Read the context carefully and answer the question by extracting the exact answer span from the context. Output only the minimal text from the context that answers the question. If the question cannot be answered based on the context, output exactly 'No answer'."""

    train_examples, validation_examples, test_examples = get_squad_v2_data(train_num, val_num, test_num)
    initial_prompt = squad_v2_initial_prompt
    
    return train_examples, validation_examples, test_examples, initial_prompt

In [3]:
LABEL_MAP = {0: "truth", 1: "lie"}

train_examples, validation_examples, test_examples, initial_prompt = data_fabric(
    'squad_v2',
    train_num=SQUAD_TRAIN_NUM,
    val_num=SQUAD_VAL_NUM,
    test_num=SQUAD_TEST_NUM,
)

print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")

Dataset prepared:
  Train: 40 examples
  Validation: 40 examples
  Test: 40 examples


In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Read the context carefully and answer the question by extracting the exact answer span from the context. Output only the minimal text from the context that answers the question. If the question cannot be answered based on the context, output exactly 'No answer'.
------------------------------------------------------------


In [5]:
TASK_DESCRIPTION = (
    "Extractive question answering: given a context paragraph and a question, "
    "extract the exact answer span from the context. If the question is unanswerable based "
    "on the context, output exactly 'No answer'."
)

optimizer = HierarchicalOptimizer(
    metrics_config=SQUAD_METRICS,
    task_description=TASK_DESCRIPTION,
)

In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=ab48b463fef6 len=262 chars
[diag] Prompt text: Read the context carefully and answer the question by extracting the exact answer span from the context. Output only the minimal text from the context that answers the question. If the question cannot be answered based on the context, output exactly 'No answer'.
[diag] evaluate_node: node_id=3881662f-ea33-44f8-9f95-bf2ace0a0d0c gen=0 source=initial split=validation stage=1 examples=40 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=40 will_use=40 weights={exact_match:0.25, no_answer_accuracy:0.2, token_f1:0.25} llm_calls_at_start=0
[diag] execute_prompt_batch: total=40 batch_size=50 n_batches=1 llm_calls_before=0
[diag]   batch 1/1: OK (40/40 done, llm_calls=0)
[diag] execute_prompt_batch done: llm_calls_delta=0 total_calls=0
[diag] evaluate_prompt execution done: llm_calls_for_execution=0
[diag]   example[0] expected='full union' actual='No answer'

  0%|          | 0/20 [00:00<?, ?it/s]

Falling back to single execution for example 1/20


  5%|▌         | 1/20 [00:00<00:18,  1.01it/s]

Falling back to single execution for example 2/20


...

Falling back to single execution for example 19/20


 95%|█████████▌| 19/20 [00:15<00:00,  1.18it/s]

Falling back to single execution for example 20/20


100%|██████████| 20/20 [00:16<00:00,  1.18it/s]


[diag] execute_prompt_batch done: llm_calls_delta=20 total_calls=136
[diag] evaluate_prompt execution done: llm_calls_for_execution=20
[diag]   example[0] expected='important trade center' actual='Answer: trade center'
[diag]   example[1] expected='disparaging individual decision-making' actual='By disparaging individual decision-making, the religion's leaders cultivate a sy'
[diag]   example[2] expected='No answer' actual='The year number was set using the Hegira year.'
[diag]   ... and 17 more examples
[diag]   metric 'semantic_similarity' skipped (stage 3 > 1)
[diag]   metric 'faithfulness' skipped (stage 3 > 1)
[diag]   metric 'robustness' skipped (stage 3 > 1)
[diag]   metric 'exact_match': 0.2000 (stage=1 weight=0.25 time=0.00s llm_calls=136)
[diag]   metric 'token_f1': 0.4180 (stage=1 weight=0.25 time=0.00s llm_calls=136)
[diag]   metric 'no_answer_accuracy': 0.7500 (stage=1 weight=0.2 time=0.00s llm_calls=136)
[diag] eval [evaluate_prompt]: stage=1 composite=0.4350 metrics=[exa

In [7]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 166.70s, best_score 0.567
  Generation 2: time 538.99s, best_score 0.584
  Generation 3: time 346.15s, best_score 0.588
  Generation 4: time 727.24s, best_score 0.590
  Generation 5: time 302.18s, best_score 0.600
Local optimizer summary:
  Total iterations recorded: 24
  Avg iteration time: 80.41s
  Total LLM calls attributed to local iterations: 354
Per-iteration breakdown:
  Iter 1: time 64.28s, llm_calls 11
  Iter 2: time 100.58s, llm_calls 19
  Iter 1: time 79.05s, llm_calls 14
  Iter 2: time 79.25s, llm_calls 15
  Iter 1: time 73.41s, llm_calls 13
  Iter 2: time 31.19s, llm_calls 6
  Iter 1: time 82.84s, llm_calls 14
  Iter 2: time 113.87s, llm_calls 37
  Iter 1: time 72.29s, llm_calls 11
  Iter 2: time 93.53s, llm_calls 17
  Iter 1: time 65.95s, llm_calls 10
  Iter 2: time 112.84s, llm_calls 22
  Iter 1: time 71.92s, llm_calls 12
  Iter 2: time 70.52s, llm_calls 15
  Iter 1: time 83.65s, llm_calls 11
  Iter 2: time 98.01s, l

In [8]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
Given a context paragraph and a question, extract the exact answer span from the context. Identify key phrases or keywords in the context that directly address the question. Limit the response to a specific character count to encourage precision. If the question is unanswerable based on the context, output exactly 'No answer'. Ensure thorough analysis of the context to extract the most relevant information.
Score: 0.600
Generation: 6
Source: local


In [9]:
print("BASELINE vs OPTIMIZED — Test Set")
print("=" * 80)

comparison = optimizer.compare_with_baseline(initial_prompt, test_examples)

print()
print("Summary:")
print(f"  {'Metric':<25s}  {'Baseline':>10s}  {'Optimized':>10s}  {'Delta':>10s}")
print(f"  {'-'*25}  {'-'*10}  {'-'*10}  {'-'*10}")
for name in ["exact_match", "token_f1", "no_answer_accuracy", "semantic_similarity", "faithfulness", "robustness"]:
   b = comparison["baseline"].get(name, 0.0)
   o = comparison["optimized"].get(name, 0.0)
   d = comparison["improvements"].get(name, 0.0)
   print(f"  {name:<25s}  {b:10.3f}  {o:10.3f}  {d:+10.3f}")
b_c = comparison["baseline"]["composite_score"]
o_c = comparison["optimized"]["composite_score"]
print(f"  {'COMPOSITE':<25s}  {b_c:10.3f}  {o_c:10.3f}  {o_c - b_c:+10.3f}")

BASELINE vs OPTIMIZED — Test Set

Comparing with baseline...
[diag] evaluate_prompt: execute=True sample=True stage=3 incoming=40 will_use=40 weights={exact_match:0.25, faithfulness:0.1, no_answer_accuracy:0.2, robustness:0.1, semantic_similarity:0.1, token_f1:0.25} llm_calls_at_start=390
[diag] execute_prompt_batch: total=40 batch_size=50 n_batches=1 llm_calls_before=390
[diag]   batch 1/1: OK (40/40 done, llm_calls=391)
[diag] execute_prompt_batch done: llm_calls_delta=1 total_calls=391
[diag] evaluate_prompt execution done: llm_calls_for_execution=1
[diag]   example[0] expected='France' actual='France'
[diag]   example[1] expected='10th and 11th centuries' actual='10th and 11th centuries'
[diag]   example[2] expected='Denmark, Iceland and Norway' actual='Denmark, Iceland and Norway'
[diag]   ... and 37 more examples
[diag]   metric 'exact_match': 0.2250 (stage=1 weight=0.25 time=0.00s llm_calls=391)
[diag]   metric 'token_f1': 0.3167 (stage=1 weight=0.25 time=0.00s llm_calls=391)
[d

In [10]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.567      | 0.567       | +0.014 ████████████████████████████
   2       | 0.584      | 0.584       | +0.017 █████████████████████████████
   3       | 0.588      | 0.588       | +0.005 █████████████████████████████
   4       | 0.590      | 0.590       | +0.002 █████████████████████████████
   5       | 0.600      | 0.600       | +0.010 █████████████████████████████




In [11]:
ыreport = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")


OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 2081.26s
   Generations: 5
   Total nodes explored: 75
Component Statistics:
   Local optimizer iterations: 24
   Local improvements: 9
   Global optimizer steps: 2
   Successful global changes: 1
Best Global Strategies:
   1. Score 0.589
      crossover...
   2. Score 0.589
      meta-optimizer...
   3. Score 0.587
      crossover...


In [12]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("="*80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:  
        print("  ↓")


EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.553
  ↓
Step 1: Generation 1, Source: local
  Score: 0.555
  Operations:
    - Added explicit instruction to verify the presence of the answer in the context before extraction....
  ↓
Step 2: Generation 2, Source: local
  Score: 0.567
  Operations:
    - Added a clear directive to output 'No answer' when the question cannot be answered....
  ↓
Step 3: Generation 2, Source: global
  Score: 0.581
  Operations:
    - Meta-optimizer (gen 2)...
  ↓
Step 4: Generation 3, Source: local
  Score: 0.587
  Operations:
    - Added specific guidance on identifying key phrases for answer extraction....
  ↓
Step 5: Generation 4, Source: local
  Score: 0.588
  Operations:
    - Added a specific instruction to output 'No answer' when the question is unanswerable....
  ↓
Step 6: Generation 5, Source: local
  Score: 0.590
  Operations:
    - Added step-by-step reasoning cues for better understanding and accuracy....
  ↓
Step 7: Ge

In [13]:
print("FINAL SUMMARY")
print("="*80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("Optimization complete!")
print(f"   Results saved to: ./optimization_results/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 75
  Evaluations performed: 75
  Generations completed: 8
  Best score achieved: 0.600
  Average score: 0.532
Local Optimization:
  Total iterations: 24
  Improvements found: 9
  Success rate: 37.5%
Global Optimization:
  Total global steps: 2
  Candidates generated: 10
  Successful changes: 1
  Success rate: 10.0%
Optimization complete!
   Results saved to: ./optimization_results/
